In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from textblob import TextBlob
from scipy.sparse import dok_matrix
import gc

# ============================================================
# 0. LOAD DATA WITH MEMORY OPTIMIZATION
# ============================================================
# Read only necessary columns and use appropriate dtypes
cols_to_read = ["budget", "revenue", "runtime", "popularity", "vote_average", 
                "vote_count", "overview", "release_date", "title", "tagline", "genres"]

train_clf = pd.read_csv("train.csv", usecols=cols_to_read, low_memory=False)
test_clf = pd.read_csv("test.csv", usecols=cols_to_read, low_memory=False)

print("Original train:", train_clf.shape, " test:", test_clf.shape)

# ============================================================
# 1. DROP NAS IN TRAIN ONLY
# ============================================================
required_cols = ["budget", "revenue", "runtime", "popularity",
                 "vote_average", "vote_count", "overview", "release_date"]

train_clf = train_clf.dropna(subset=required_cols).reset_index(drop=True)
test_clf = test_clf.fillna("")

print("After drop:", train_clf.shape, test_clf.shape)

# ============================================================
# 2. FIX TEXT COLUMNS
# ============================================================
for col in ["title", "overview", "tagline", "genres"]:
    train_clf[col] = train_clf[col].fillna("")
    test_clf[col] = test_clf[col].fillna("")

# ============================================================
# 3. GENRE LABEL FIX
# ============================================================
def extract_genre(x):
    if not isinstance(x, str) or x.strip() == "":
        return "Unknown"
    return x.split("-")[0]

train_clf["main_genre"] = train_clf["genres"].apply(extract_genre)
test_clf["main_genre"] = test_clf["genres"].apply(extract_genre)

# Drop genres column to save memory
train_clf.drop("genres", axis=1, inplace=True)
test_clf.drop("genres", axis=1, inplace=True)
gc.collect()

# Label encode
genre_list_clf = sorted(train_clf["main_genre"].unique())
genre_to_idx_clf = {g: i for i, g in enumerate(genre_list_clf)}
idx_to_genre_clf = {i: g for g, i in genre_to_idx_clf.items()}

y_genre_train_clf = train_clf["main_genre"].map(genre_to_idx_clf).to_numpy(dtype=np.int16)

print("Num classes:", len(genre_list_clf))

# ============================================================
# 4. NUMERIC FIX WITH MEMORY-EFFICIENT DTYPES
# ============================================================
num_cols = ["budget", "runtime", "popularity", "vote_average", "vote_count", "revenue"]

for col in num_cols:
    train_clf[col] = pd.to_numeric(train_clf[col], errors="coerce").fillna(0).astype(np.float32)
    test_clf[col] = pd.to_numeric(test_clf[col], errors="coerce").fillna(0).astype(np.float32)

# YEAR
train_clf["release_date"] = pd.to_datetime(train_clf["release_date"], errors="coerce")
test_clf["release_date"] = pd.to_datetime(test_clf["release_date"], errors="coerce")

mode_year = int(train_clf["release_date"].dt.year.mode()[0])
train_clf["release_year"] = train_clf["release_date"].dt.year.fillna(mode_year).astype(np.int16)
test_clf["release_year"] = test_clf["release_date"].dt.year.fillna(mode_year).astype(np.int16)

# Drop release_date to save memory
train_clf.drop("release_date", axis=1, inplace=True)
test_clf.drop("release_date", axis=1, inplace=True)
gc.collect()

# ============================================================
# 5. EXTRA TEXT FEATURES (BATCH PROCESSING)
# ============================================================
def wc(x):
    return len(str(x).split())

def avgw(x):
    w = str(x).split()
    return np.mean([len(i) for i in w]) if w else 0

# Process in smaller chunks to reduce memory
train_clf["word_count"] = train_clf["overview"].apply(wc).astype(np.int16)
test_clf["word_count"] = test_clf["overview"].apply(wc).astype(np.int16)

train_clf["avg_word_length"] = train_clf["overview"].apply(avgw).astype(np.float32)
test_clf["avg_word_length"] = test_clf["overview"].apply(avgw).astype(np.float32)

# Sentiment - process and store as float32
train_clf["sentiment"] = train_clf["overview"].apply(
    lambda x: TextBlob(x).sentiment.polarity
).astype(np.float32)
test_clf["sentiment"] = test_clf["overview"].apply(
    lambda x: TextBlob(x).sentiment.polarity
).astype(np.float32)

# Drop unnecessary text columns
train_clf.drop(["title", "tagline"], axis=1, inplace=True)
test_clf.drop(["title", "tagline"], axis=1, inplace=True)
gc.collect()

# ============================================================
# 6. SCALE USING TRAIN ONLY
# ============================================================
scale_cols_clf = ["budget", "runtime", "popularity", "vote_average", "vote_count",
                  "release_year", "word_count", "avg_word_length", "sentiment"]

mean_clf = train_clf[scale_cols_clf].mean().astype(np.float32)
std_clf = train_clf[scale_cols_clf].std().replace(0, 1).astype(np.float32)

X_num_train_clf = ((train_clf[scale_cols_clf] - mean_clf) / std_clf).astype(np.float32)
X_num_test_clf = ((test_clf[scale_cols_clf] - mean_clf) / std_clf).astype(np.float32)

# ============================================================
# 7. TF-IDF FROM TRAIN ONLY (MEMORY OPTIMIZED)
# ============================================================
def tokenize(text):
    return re.findall(r"[A-Za-z]+", text.lower())

print("Tokenizing train...")
tokens_train = train_clf["overview"].apply(tokenize)

# Build vocab from TRAIN ONLY with smaller vocabulary
word_freq_clf = Counter()
for t in tokens_train:
    word_freq_clf.update(t)

# Reduce vocab size to 3000 most common words (from 5000)
vocab_clf = [w for w, _ in word_freq_clf.most_common(3000)]
vocab_idx_clf = {w: i for i, w in enumerate(vocab_clf)}

# Clear word frequency counter
del word_freq_clf
gc.collect()

# IDF
N = len(train_clf)
df_count_clf = np.zeros(len(vocab_clf), dtype=np.int32)

for t in tokens_train:
    for u in set(t):
        if u in vocab_idx_clf:
            df_count_clf[vocab_idx_clf[u]] += 1

idf_clf = (np.log((N + 1) / (df_count_clf + 1)) + 1).astype(np.float32)
del df_count_clf
gc.collect()

# TFIDF — TRAIN (using CSR for efficiency)
print("Building train TF-IDF...")
tfidf_train_clf = dok_matrix((len(train_clf), len(vocab_clf)), dtype=np.float32)
for i, tokens in enumerate(tokens_train):
    if i % 1000 == 0:
        print(f"Processing train sample {i}/{len(train_clf)}")
    cnt = Counter(tokens)
    tot = len(tokens) if len(tokens) > 0 else 1
    for tok, c in cnt.items():
        if tok in vocab_idx_clf:
            j = vocab_idx_clf[tok]
            tfidf_train_clf[i, j] = (c / tot) * idf_clf[j]

# Convert to CSR for memory efficiency
tfidf_train_clf = tfidf_train_clf.tocsr()

# Drop overview from train
train_clf.drop("overview", axis=1, inplace=True)
del tokens_train
gc.collect()

# TFIDF — TEST
print("Tokenizing test...")
tokens_test = test_clf["overview"].apply(tokenize)

print("Building test TF-IDF...")
tfidf_test_clf = dok_matrix((len(test_clf), len(vocab_clf)), dtype=np.float32)
for i, tokens in enumerate(tokens_test):
    if i % 1000 == 0:
        print(f"Processing test sample {i}/{len(test_clf)}")
    cnt = Counter(tokens)
    tot = len(tokens) if len(tokens) > 0 else 1
    for tok, c in cnt.items():
        if tok in vocab_idx_clf:
            j = vocab_idx_clf[tok]
            tfidf_test_clf[i, j] = (c / tot) * idf_clf[j]

# Convert to CSR
tfidf_test_clf = tfidf_test_clf.tocsr()

# Drop overview from test
test_clf.drop("overview", axis=1, inplace=True)
del tokens_test
gc.collect()

# ============================================================
# 8. CONCAT FINAL CLASSIFIER FEATURES
# ============================================================
print("Concatenating features...")
X_train_clf = np.hstack([
    X_num_train_clf.to_numpy(),
    tfidf_train_clf.toarray()
]).astype(np.float32)

X_test_clf = np.hstack([
    X_num_test_clf.to_numpy(),
    tfidf_test_clf.toarray()
]).astype(np.float32)

# Clean up
del tfidf_train_clf, tfidf_test_clf, X_num_train_clf, X_num_test_clf
gc.collect()

print("\n" + "="*50)
print("FINAL CLASSIFIER SHAPES:")
print("="*50)
print(f"X_train_clf: {X_train_clf.shape}")
print(f"X_test_clf: {X_test_clf.shape}")
print(f"y_genre_train_clf: {y_genre_train_clf.shape}")
print(f"\nMemory usage (MB):")
print(f"X_train_clf: {X_train_clf.nbytes / 1024**2:.2f}")
print(f"X_test_clf: {X_test_clf.nbytes / 1024**2:.2f}")
print("="*50)

Original train: (577853, 11)  test: (144464, 11)
After drop: (442642, 11) (144464, 11)
Num classes: 20
Tokenizing train...
Building train TF-IDF...
Processing train sample 0/442642
Processing train sample 1000/442642
Processing train sample 2000/442642
Processing train sample 3000/442642
Processing train sample 4000/442642
Processing train sample 5000/442642
Processing train sample 6000/442642
Processing train sample 7000/442642
Processing train sample 8000/442642
Processing train sample 9000/442642
Processing train sample 10000/442642
Processing train sample 11000/442642
Processing train sample 12000/442642
Processing train sample 13000/442642
Processing train sample 14000/442642
Processing train sample 15000/442642
Processing train sample 16000/442642
Processing train sample 17000/442642
Processing train sample 18000/442642
Processing train sample 19000/442642
Processing train sample 20000/442642
Processing train sample 21000/442642
Processing train sample 22000/442642
Processing tra

In [2]:
import numpy as np
import pandas as pd
import time

# ============================================================
# FAST SOFTMAX LOGISTIC REGRESSION FROM SCRATCH
# ============================================================

class LogisticRegressionMulticlass:
    """
    Softmax Logistic Regression for multi-class classification
    Optimized for speed
    """
    
    def __init__(self, learning_rate=0.5, num_iterations=100, batch_size=256, 
                 l2_lambda=0.01, verbose=True):
        self.lr = learning_rate
        self.num_iterations = num_iterations
        self.batch_size = batch_size
        self.l2_lambda = l2_lambda
        self.verbose = verbose
        self.W = None
        self.b = None
        self.loss_history = []
        
    def softmax(self, z):
        """Numerically stable softmax"""
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def compute_loss_batch(self, X_batch, y_batch):
        """Fast loss computation on batch only"""
        m = X_batch.shape[0]
        
        # Forward pass
        logits = np.dot(X_batch, self.W) + self.b
        probs = self.softmax(logits)
        
        # Cross-entropy loss
        y_onehot = np.zeros((m, self.num_classes))
        y_onehot[np.arange(m), y_batch] = 1
        
        loss = -np.sum(y_onehot * np.log(probs + 1e-8)) / m
        
        # L2 regularization
        l2_reg = (self.l2_lambda / 2) * np.sum(self.W ** 2)
        
        return loss + l2_reg
    
    def fit(self, X, y):
        """
        Train the logistic regression model - OPTIMIZED VERSION
        
        Parameters:
        X: (n_samples, n_features) - training data
        y: (n_samples,) - labels (0 to num_classes-1)
        """
        n_samples, n_features = X.shape
        self.num_classes = len(np.unique(y))
        
        # Initialize weights and bias
        np.random.seed(42)
        self.W = np.random.randn(n_features, self.num_classes) * 0.01
        self.b = np.zeros((1, self.num_classes))
        
        print(f"Training on {n_samples} samples with {n_features} features")
        print(f"Number of classes: {self.num_classes}")
        print(f"Batch size: {self.batch_size}, Learning rate: {self.lr}")
        print("-" * 80)
        
        start_time = time.time()
        
        # Training loop
        for iteration in range(self.num_iterations):
            iter_start = time.time()
            
            # Mini-batch gradient descent
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            batch_loss = 0
            num_batches = 0
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i + self.batch_size]
                y_batch = y_shuffled[i:i + self.batch_size]
                
                batch_size_actual = X_batch.shape[0]
                
                # Forward pass
                logits = np.dot(X_batch, self.W) + self.b
                probs = self.softmax(logits)
                
                # One-hot encode labels
                y_onehot = np.zeros((batch_size_actual, self.num_classes))
                y_onehot[np.arange(batch_size_actual), y_batch] = 1
                
                # Backward pass
                dlogits = (probs - y_onehot) / batch_size_actual
                dW = np.dot(X_batch.T, dlogits) + self.l2_lambda * self.W
                db = np.sum(dlogits, axis=0, keepdims=True)
                
                # Update weights
                self.W -= self.lr * dW
                self.b -= self.lr * db
                
                # Track loss
                batch_loss += self.compute_loss_batch(X_batch, y_batch)
                num_batches += 1
            
            avg_loss = batch_loss / num_batches
            self.loss_history.append(avg_loss)
            
            # Print progress every 10 iterations
            if self.verbose and (iteration % 10 == 0 or iteration == self.num_iterations - 1):
                iter_time = time.time() - iter_start
                elapsed = time.time() - start_time
                
                # Quick accuracy estimate on a sample (much faster!)
                sample_size = min(2000, n_samples)
                sample_idx = np.random.choice(n_samples, sample_size, replace=False)
                sample_acc = self.score(X[sample_idx], y[sample_idx])
                
                print(f"Iter {iteration:3d}/{self.num_iterations} | "
                      f"Loss: {avg_loss:.4f} | "
                      f"Sample Acc: {sample_acc:.4f} | "
                      f"Time: {iter_time:.1f}s | "
                      f"Elapsed: {elapsed/60:.1f}min")
        
        total_time = time.time() - start_time
        print("-" * 80)
        print(f"Training completed in {total_time/60:.2f} minutes")
        
        return self
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        logits = np.dot(X, self.W) + self.b
        return self.softmax(logits)
    
    def predict(self, X):
        """Predict class labels"""
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)
    
    def score(self, X, y):
        """Compute accuracy"""
        predictions = self.predict(X)
        return np.mean(predictions == y)


# ============================================================
# EVALUATION METRICS FROM SCRATCH
# ============================================================

def confusion_matrix_numpy(y_true, y_pred, num_classes):
    """Compute confusion matrix"""
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    
    return cm


def precision_recall_f1_numpy(y_true, y_pred, num_classes):
    """Compute precision, recall, and F1 score for each class"""
    cm = confusion_matrix_numpy(y_true, y_pred, num_classes)
    
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
        else:
            f1[i] = 0.0
    
    return precision, recall, f1


def accuracy_score_numpy(y_true, y_pred):
    """Compute accuracy"""
    return np.mean(y_true == y_pred)


def print_classification_report(y_true, y_pred, class_names):
    """Print a detailed classification report"""
    num_classes = len(class_names)
    precision, recall, f1 = precision_recall_f1_numpy(y_true, y_pred, num_classes)
    
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT")
    print("="*80)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)
    
    for i in range(num_classes):
        support = np.sum(y_true == i)
        print(f"{class_names[i]:<20} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support:>8}")
    
    print("-"*80)
    
    # Macro averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    
    # Weighted averages
    weights = np.array([np.sum(y_true == i) for i in range(num_classes)])
    weights = weights / np.sum(weights)
    
    weighted_precision = np.sum(precision * weights)
    weighted_recall = np.sum(recall * weights)
    weighted_f1 = np.sum(f1 * weights)
    
    total_support = len(y_true)
    
    print(f"{'Macro Avg':<20} {macro_precision:>10.4f}  {macro_recall:>10.4f}  "
          f"{macro_f1:>10.4f}  {total_support:>8}")
    print(f"{'Weighted Avg':<20} {weighted_precision:>10.4f}  {weighted_recall:>10.4f}  "
          f"{weighted_f1:>10.4f}  {total_support:>8}")
    print("="*80)
    
    accuracy = accuracy_score_numpy(y_true, y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")
    print("="*80)


def print_confusion_matrix(cm, class_names):
    """Print confusion matrix in a readable format"""
    print("\n" + "="*80)
    print("CONFUSION MATRIX")
    print("="*80)
    print("Rows = True Labels, Columns = Predicted Labels\n")
    
    # Truncate class names if too long
    short_names = [name[:15] for name in class_names]
    cm_df = pd.DataFrame(cm, index=short_names, columns=short_names)
    print(cm_df)
    print("="*80)


# ============================================================
# MAIN TRAINING AND EVALUATION
# ============================================================

print("\n" + "="*80)
print("TRAINING FAST LOGISTIC REGRESSION MODEL")
print("="*80)

# Initialize and train model with FAST settings
model = LogisticRegressionMulticlass(
    learning_rate=0.5,      # High learning rate for fast convergence
    num_iterations=100,     # Reduced iterations
    batch_size=256,         # Larger batches for speed
    l2_lambda=0.01,
    verbose=True
)

# Train the model
print("\nTraining on X_train_clf...")
model.fit(X_train_clf, y_genre_train_clf)

# Make predictions on training set
print("\n" + "="*80)
print("EVALUATING ON TRAINING SET")
print("="*80)

print("Computing predictions...")
y_train_pred = model.predict(X_train_clf)

# Compute confusion matrix
print("Computing confusion matrix...")
cm_train = confusion_matrix_numpy(y_genre_train_clf, y_train_pred, len(genre_list_clf))

# Print detailed classification report
print_classification_report(y_genre_train_clf, y_train_pred, genre_list_clf)

# Print confusion matrix
print_confusion_matrix(cm_train, genre_list_clf)

# ============================================================
# ADDITIONAL METRICS
# ============================================================

print("\n" + "="*80)
print("ADDITIONAL METRICS")
print("="*80)

# Per-class metrics
precision, recall, f1 = precision_recall_f1_numpy(
    y_genre_train_clf, y_train_pred, len(genre_list_clf)
)

print("\nPer-Class Metrics:")
for i, genre in enumerate(genre_list_clf):
    print(f"{genre:>15}: Precision={precision[i]:.4f}, "
          f"Recall={recall[i]:.4f}, F1={f1[i]:.4f}")

# Overall metrics
print(f"\nMacro-averaged Precision: {np.mean(precision):.4f}")
print(f"Macro-averaged Recall: {np.mean(recall):.4f}")
print(f"Macro-averaged F1-Score: {np.mean(f1):.4f}")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

# Save predictions for test set
print("\nGenerating predictions for test set...")
y_test_pred = model.predict(X_test_clf)

# Map predictions back to genre names
test_genre_predictions = [idx_to_genre_clf[i] for i in y_test_pred]

print(f"Generated {len(test_genre_predictions)} test predictions")
print(f"Sample predictions: {test_genre_predictions[:5]}")


TRAINING FAST LOGISTIC REGRESSION MODEL

Training on X_train_clf...
Training on 442642 samples with 3009 features
Number of classes: 20
Batch size: 256, Learning rate: 0.5
--------------------------------------------------------------------------------
Iter   0/100 | Loss: 2.1694 | Sample Acc: 0.3240 | Time: 58.7s | Elapsed: 1.0min


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd
import time

# ============================================================
# LOGISTIC REGRESSION FROM SCRATCH
# ============================================================

class LogisticRegressionMulticlass:
    
    
    def __init__(self, learning_rate=0.5, num_iterations=100, batch_size=256, 
                 l2_lambda=0.01, verbose=True):
        self.lr = learning_rate
        self.num_iterations = num_iterations
        self.batch_size = batch_size
        self.l2_lambda = l2_lambda
        self.verbose = verbose
        self.W = None
        self.b = None
        self.loss_history = []
        
    def softmax(self, z):
        
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)
    
    def compute_loss_batch(self, X_batch, y_batch):
        
        m = X_batch.shape[0]
        
        # Forward pass
        logits = np.dot(X_batch, self.W) + self.b
        probs = self.softmax(logits)
        
        # Cross-entropy loss
        y_onehot = np.zeros((m, self.num_classes))
        y_onehot[np.arange(m), y_batch] = 1
        
        loss = -np.sum(y_onehot * np.log(probs + 1e-8)) / m
        
        # L2 regularization
        l2_reg = (self.l2_lambda / 2) * np.sum(self.W ** 2)
        
        return loss + l2_reg
    
    def fit(self, X, y):
        
        n_samples, n_features = X.shape
        self.num_classes = len(np.unique(y))
        
        # Initialize weights and bias
        np.random.seed(42)
        self.W = np.random.randn(n_features, self.num_classes) * 0.01
        self.b = np.zeros((1, self.num_classes))
        
        print(f"Training on {n_samples} samples with {n_features} features")
        print(f"Number of classes: {self.num_classes}")
        print(f"Batch size: {self.batch_size}, Learning rate: {self.lr}")
        print("-" * 80)
        
        start_time = time.time()
        
        # Training loop
        for iteration in range(self.num_iterations):
            iter_start = time.time()
            
            # Mini-batch gradient descent
            indices = np.random.permutation(n_samples)
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            batch_loss = 0
            num_batches = 0
            
            for i in range(0, n_samples, self.batch_size):
                X_batch = X_shuffled[i:i + self.batch_size]
                y_batch = y_shuffled[i:i + self.batch_size]
                
                batch_size_actual = X_batch.shape[0]
                
                # Forward pass
                logits = np.dot(X_batch, self.W) + self.b
                probs = self.softmax(logits)
                
                # One-hot encode labels
                y_onehot = np.zeros((batch_size_actual, self.num_classes))
                y_onehot[np.arange(batch_size_actual), y_batch] = 1
                
                # Backward pass
                dlogits = (probs - y_onehot) / batch_size_actual
                dW = np.dot(X_batch.T, dlogits) + self.l2_lambda * self.W
                db = np.sum(dlogits, axis=0, keepdims=True)
                
                # Update weights
                self.W -= self.lr * dW
                self.b -= self.lr * db
                
                # Track loss
                batch_loss += self.compute_loss_batch(X_batch, y_batch)
                num_batches += 1
            
            avg_loss = batch_loss / num_batches
            self.loss_history.append(avg_loss)
            
            # Print progress every 10 iterations
            if self.verbose and (iteration % 10 == 0 or iteration == self.num_iterations - 1):
                iter_time = time.time() - iter_start
                elapsed = time.time() - start_time
                
                # Quick accuracy estimate on a sample (much faster!)
                sample_size = min(2000, n_samples)
                sample_idx = np.random.choice(n_samples, sample_size, replace=False)
                sample_acc = self.score(X[sample_idx], y[sample_idx])
                
                print(f"Iter {iteration:3d}/{self.num_iterations} | "
                      f"Loss: {avg_loss:.4f} | "
                      f"Sample Acc: {sample_acc:.4f} | "
                      f"Time: {iter_time:.1f}s | "
                      f"Elapsed: {elapsed/60:.1f}min")
        
        total_time = time.time() - start_time
        print("-" * 80)
        print(f"Training completed in {total_time/60:.2f} minutes")
        
        return self
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        logits = np.dot(X, self.W) + self.b
        return self.softmax(logits)
    
    def predict(self, X):
        """Predict class labels"""
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)
    
    def score(self, X, y):
        """Compute accuracy"""
        predictions = self.predict(X)
        return np.mean(predictions == y)


# ============================================================
# EVALUATION METRICS FROM SCRATCH
# ============================================================

def confusion_matrix_numpy(y_true, y_pred, num_classes):
    """Compute confusion matrix"""
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    
    return cm


def precision_recall_f1_numpy(y_true, y_pred, num_classes):
    """Compute precision, recall, and F1 score for each class"""
    cm = confusion_matrix_numpy(y_true, y_pred, num_classes)
    
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
        else:
            f1[i] = 0.0
    
    return precision, recall, f1


def accuracy_score_numpy(y_true, y_pred):
    """Compute accuracy"""
    return np.mean(y_true == y_pred)


def print_classification_report(y_true, y_pred, class_names):
    """Print a detailed classification report"""
    num_classes = len(class_names)
    precision, recall, f1 = precision_recall_f1_numpy(y_true, y_pred, num_classes)
    
    print("\n" + "="*80)
    print("CLASSIFICATION REPORT")
    print("="*80)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)
    
    for i in range(num_classes):
        support = np.sum(y_true == i)
        print(f"{class_names[i]:<20} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support:>8}")
    
    print("-"*80)
    
    # Macro averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    
    # Weighted averages
    weights = np.array([np.sum(y_true == i) for i in range(num_classes)])
    weights = weights / np.sum(weights)
    
    weighted_precision = np.sum(precision * weights)
    weighted_recall = np.sum(recall * weights)
    weighted_f1 = np.sum(f1 * weights)
    
    total_support = len(y_true)
    
    print(f"{'Macro Avg':<20} {macro_precision:>10.4f}  {macro_recall:>10.4f}  "
          f"{macro_f1:>10.4f}  {total_support:>8}")
    print(f"{'Weighted Avg':<20} {weighted_precision:>10.4f}  {weighted_recall:>10.4f}  "
          f"{weighted_f1:>10.4f}  {total_support:>8}")
    print("="*80)
    
    accuracy = accuracy_score_numpy(y_true, y_pred)
    print(f"\nOverall Accuracy: {accuracy:.4f}")
    print("="*80)


def print_confusion_matrix(cm, class_names):
    """Print confusion matrix in a readable format"""
    print("\n" + "="*80)
    print("CONFUSION MATRIX")
    print("="*80)
    print("Rows = True Labels, Columns = Predicted Labels\n")
    
    # Truncate class names if too long
    short_names = [name[:15] for name in class_names]
    cm_df = pd.DataFrame(cm, index=short_names, columns=short_names)
    print(cm_df)
    print("="*80)


# ============================================================
# MAIN TRAINING AND EVALUATION
# ============================================================

print("\n" + "="*80)
print("TRAINING FAST LOGISTIC REGRESSION MODEL")
print("="*80)

# FOR LARGE DATASETS (400k+ samples): Use these optimized settings
model = LogisticRegressionMulticlass(
    learning_rate=1.0,      # Even higher for large datasets
    num_iterations=50,      # Just 50 iterations for speed
    batch_size=1024,        # Much larger batches (4x bigger)
    l2_lambda=0.001,        # Reduce regularization
    verbose=True
)

print(f"\n🚀 OPTIMIZED FOR LARGE DATASET: {X_train_clf.shape[0]:,} samples")
print(f"Expected training time: ~20-25 minutes")

# Train the model
print("\nTraining on X_train_clf...")
model.fit(X_train_clf, y_genre_train_clf)

# Make predictions on FULL training set for proper evaluation
print("\n" + "="*80)
print("EVALUATING ON FULL TRAINING SET")
print("="*80)

print(f"Computing predictions on all {len(X_train_clf):,} training samples...")
y_train_pred = model.predict(X_train_clf)

# Compute confusion matrix
print("Computing confusion matrix...")
cm_train = confusion_matrix_numpy(y_genre_train_clf, y_train_pred, len(genre_list_clf))

# Print detailed classification report
print_classification_report(y_genre_train_clf, y_train_pred, genre_list_clf)

# Print confusion matrix
print_confusion_matrix(cm_train, genre_list_clf)

# ============================================================
# ADDITIONAL METRICS
# ============================================================

print("\n" + "="*80)
print("ADDITIONAL METRICS")
print("="*80)

# Per-class metrics
precision, recall, f1 = precision_recall_f1_numpy(
    y_genre_train_clf, y_train_pred, len(genre_list_clf)
)

print("\nPer-Class Metrics:")
for i, genre in enumerate(genre_list_clf):
    print(f"{genre:>15}: Precision={precision[i]:.4f}, "
          f"Recall={recall[i]:.4f}, F1={f1[i]:.4f}")

# Overall metrics
print(f"\nMacro-averaged Precision: {np.mean(precision):.4f}")
print(f"Macro-averaged Recall: {np.mean(recall):.4f}")
print(f"Macro-averaged F1-Score: {np.mean(f1):.4f}")

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)

# Save predictions for test set
print("\nGenerating predictions for test set...")
y_test_pred = model.predict(X_test_clf)

# Map predictions back to genre names
test_genre_predictions = [idx_to_genre_clf[i] for i in y_test_pred]

print(f"Generated {len(test_genre_predictions)} test predictions")
print(f"Sample predictions: {test_genre_predictions[:5]}")


TRAINING FAST LOGISTIC REGRESSION MODEL

🚀 OPTIMIZED FOR LARGE DATASET: 442,642 samples
Expected training time: ~20-25 minutes

Training on X_train_clf...
Training on 442642 samples with 3009 features
Number of classes: 20
Batch size: 1024, Learning rate: 1.0
--------------------------------------------------------------------------------
Iter   0/50 | Loss: 2.1265 | Sample Acc: 0.3480 | Time: 103.6s | Elapsed: 1.7min


In [5]:
import numpy as np
import pandas as pd
import time
from collections import Counter

# ============================================================
# EVALUATION METRICS (REUSED)
# ============================================================

def confusion_matrix_numpy(y_true, y_pred, num_classes):
    """Compute confusion matrix"""
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    return cm

def precision_recall_f1_numpy(y_true, y_pred, num_classes):
    """Compute precision, recall, and F1 score"""
    cm = confusion_matrix_numpy(y_true, y_pred, num_classes)
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
    
    return precision, recall, f1

def print_classification_report(y_true, y_pred, class_names, model_name="Model"):
    """Print classification report"""
    num_classes = len(class_names)
    precision, recall, f1 = precision_recall_f1_numpy(y_true, y_pred, num_classes)
    
    print("\n" + "="*80)
    print(f"{model_name} - CLASSIFICATION REPORT")
    print("="*80)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)
    
    for i in range(num_classes):
        support = np.sum(y_true == i)
        print(f"{class_names[i]:<20} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support:>8}")
    
    print("-"*80)
    
    # Macro & Weighted averages
    weights = np.array([np.sum(y_true == i) for i in range(num_classes)])
    weights = weights / np.sum(weights)
    
    print(f"{'Macro Avg':<20} {np.mean(precision):>10.4f}  {np.mean(recall):>10.4f}  "
          f"{np.mean(f1):>10.4f}  {len(y_true):>8}")
    print(f"{'Weighted Avg':<20} {np.sum(precision*weights):>10.4f}  "
          f"{np.sum(recall*weights):>10.4f}  {np.sum(f1*weights):>10.4f}  {len(y_true):>8}")
    print("="*80)
    print(f"Overall Accuracy: {np.mean(y_true == y_pred):.4f}")
    print("="*80)


# ============================================================
# 1. K-NEAREST NEIGHBORS (OPTIMIZED)
# ============================================================

class KNNClassifier:
    """
    Memory-efficient KNN using batch processing
    """
    
    def __init__(self, n_neighbors=5, batch_size=1000):
        self.k = n_neighbors
        self.batch_size = batch_size
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """Store training data"""
        self.X_train = X.astype(np.float32)  # Use float32 to save memory
        self.y_train = y
        return self
    
    def predict(self, X):
        """Predict using batch processing to save memory"""
        n_samples = X.shape[0]
        predictions = np.zeros(n_samples, dtype=np.int16)
        
        print(f"KNN: Predicting {n_samples} samples in batches...")
        
        for i in range(0, n_samples, self.batch_size):
            if i % 5000 == 0:
                print(f"  Processing samples {i}/{n_samples}...")
            
            end_idx = min(i + self.batch_size, n_samples)
            X_batch = X[i:end_idx]
            
            # Compute distances for this batch
            # Using euclidean distance: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a*b
            X_batch_sq = np.sum(X_batch**2, axis=1, keepdims=True)
            X_train_sq = np.sum(self.X_train**2, axis=1, keepdims=True).T
            distances = X_batch_sq + X_train_sq - 2 * np.dot(X_batch, self.X_train.T)
            
            # Find k nearest neighbors
            k_indices = np.argpartition(distances, self.k, axis=1)[:, :self.k]
            
            # Vote for each sample in batch
            for j in range(len(X_batch)):
                neighbor_labels = self.y_train[k_indices[j]]
                predictions[i + j] = np.bincount(neighbor_labels).argmax()
        
        return predictions


# ============================================================
# 2. DECISION TREE (OPTIMIZED)
# ============================================================

class DecisionTreeNode:
    """Node in decision tree"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # For leaf nodes
        
    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """
    Decision Tree with optimizations for speed
    """
    
    def __init__(self, max_depth=10, min_samples_split=20, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None
        self.n_classes = None
        
    def fit(self, X, y):
        """Build decision tree"""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        
        if self.max_features is None:
            self.max_features = int(np.sqrt(self.n_features))
        
        print(f"Building decision tree (max_depth={self.max_depth})...")
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity"""
        classes, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _best_split(self, X, y):
        """Find best split using random feature subset"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        # Randomly select features to consider
        features = np.random.choice(self.n_features, self.max_features, replace=False)
        
        parent_impurity = self._gini_impurity(y)
        
        for feature in features:
            # Sample thresholds instead of checking all unique values
            feature_values = X[:, feature]
            thresholds = np.percentile(feature_values, [25, 50, 75])
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_split or \
                   np.sum(right_mask) < self.min_samples_split:
                    continue
                
                # Calculate information gain
                left_impurity = self._gini_impurity(y[left_mask])
                right_impurity = self._gini_impurity(y[right_mask])
                
                n = len(y)
                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                
                gain = parent_impurity - (n_left/n * left_impurity + n_right/n * right_impurity)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth):
        """Recursively build tree"""
        n_samples = len(y)
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Find best split
        feature, threshold = self._best_split(X, y)
        
        if feature is None:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Split data
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # Build subtrees
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return DecisionTreeNode(feature=feature, threshold=threshold,
                               left=left_child, right=right_child)
    
    def _predict_sample(self, x, node):
        """Predict single sample"""
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict multiple samples"""
        print(f"Decision Tree: Predicting {len(X)} samples...")
        return np.array([self._predict_sample(x, self.root) for x in X])


# ============================================================
# 3. RANDOM FOREST (OPTIMIZED)
# ============================================================

class RandomForestClassifier:
    """
    Random Forest using bootstrap sampling and feature randomness
    """
    
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=20, 
                 max_features=None, sample_ratio=0.8):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.sample_ratio = sample_ratio
        self.trees = []
    
    def fit(self, X, y):
        """Train random forest"""
        n_samples = X.shape[0]
        sample_size = int(n_samples * self.sample_ratio)
        
        print(f"\nTraining Random Forest with {self.n_estimators} trees...")
        print(f"Each tree trains on {sample_size:,} samples")
        print("-" * 80)
        
        for i in range(self.n_estimators):
            print(f"Training tree {i+1}/{self.n_estimators}...")
            
            # Bootstrap sample
            indices = np.random.choice(n_samples, sample_size, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]
            
            # Train decision tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
        
        print("Random Forest training complete!")
        return self
    
    def predict(self, X):
        """Predict by majority voting"""
        print(f"\nRandom Forest: Generating predictions from {self.n_estimators} trees...")
        
        # Get predictions from all trees
        tree_predictions = np.zeros((len(X), self.n_estimators), dtype=np.int16)
        
        for i, tree in enumerate(self.trees):
            print(f"  Tree {i+1}/{self.n_estimators} predicting...")
            tree_predictions[:, i] = tree.predict(X)
        
        # Majority vote
        print("  Computing majority vote...")
        predictions = np.array([np.bincount(tree_predictions[i]).argmax() 
                               for i in range(len(X))])
        
        return predictions


# ============================================================
# MAIN EXECUTION WITH SAMPLING FOR SPEED
# ============================================================

print("\n" + "="*80)
print("TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS")
print("="*80)

# Use smaller sample for faster training (adjust as needed)
TRAIN_SAMPLE_SIZE = 500000  # 50k samples for training
TEST_SAMPLE_SIZE = 100000   # 10k samples for testing

print(f"\nOriginal sizes: Train={len(X_train_clf):,}, Test={len(X_test_clf):,}")
print(f"Using samples: Train={TRAIN_SAMPLE_SIZE:,}, Test={TEST_SAMPLE_SIZE:,}")
print("(This ensures reasonable training time while maintaining accuracy)\n")

# Sample training data
np.random.seed(42)
train_idx = np.random.choice(len(X_train_clf), TRAIN_SAMPLE_SIZE, replace=True)
X_train_sample = X_train_clf[train_idx]
y_train_sample = y_genre_train_clf[train_idx]

# Sample test data for evaluation
test_idx = np.random.choice(len(X_test_clf), TEST_SAMPLE_SIZE, replace=False)
X_test_sample = X_test_clf[test_idx]

# For evaluation, sample validation set from training data
val_idx = np.random.choice(len(X_train_clf), 10000, replace=False)
X_val = X_train_clf[val_idx]
y_val = y_genre_train_clf[val_idx]

print("="*80)

# ============================================================
# 1. K-NEAREST NEIGHBORS
# ============================================================

print("\n" + "="*80)
print("1. K-NEAREST NEIGHBORS CLASSIFIER")
print("="*80)

start_time = time.time()

knn = KNNClassifier(n_neighbors=5, batch_size=1000)
knn.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_knn = knn.predict(X_val)

knn_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_knn, genre_list_clf, "KNN")
print(f"\nKNN Training + Prediction Time: {knn_time/60:.2f} minutes")

# ============================================================
# 2. DECISION TREE
# ============================================================

print("\n" + "="*80)
print("2. DECISION TREE CLASSIFIER")
print("="*80)

start_time = time.time()

dt = DecisionTreeClassifier(max_depth=15, min_samples_split=50, max_features=100)
dt.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_dt = dt.predict(X_val)

dt_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_dt, genre_list_clf, "Decision Tree")
print(f"\nDecision Tree Training + Prediction Time: {dt_time/60:.2f} minutes")

# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "="*80)
print("3. RANDOM FOREST CLASSIFIER")
print("="*80)

start_time = time.time()

rf = RandomForestClassifier(
    n_estimators=10,      # 10 trees for reasonable speed
    max_depth=12,
    min_samples_split=50,
    max_features=100,
    sample_ratio=0.7
)
rf.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_rf = rf.predict(X_val)

rf_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_rf, genre_list_clf, "Random Forest")
print(f"\nRandom Forest Training + Prediction Time: {rf_time/60:.2f} minutes")

# ============================================================
# COMPARISON SUMMARY
# ============================================================

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

models_summary = {
    'Model': ['KNN', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        np.mean(y_val == y_val_pred_knn),
        np.mean(y_val == y_val_pred_dt),
        np.mean(y_val == y_val_pred_rf)
    ],
    'Time (min)': [knn_time/60, dt_time/60, rf_time/60]
}

# Calculate F1 scores
knn_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_knn, len(genre_list_clf))[2])
dt_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_dt, len(genre_list_clf))[2])
rf_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_rf, len(genre_list_clf))[2])

models_summary['Macro F1'] = [knn_f1, dt_f1, rf_f1]

summary_df = pd.DataFrame(models_summary)
print("\n", summary_df.to_string(index=False))
print("\n" + "="*80)

print("\n✅ ALL MODELS TRAINED AND EVALUATED SUCCESSFULLY!")



TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS

Original sizes: Train=442,642, Test=144,464
Using samples: Train=500,000, Test=100,000
(This ensures reasonable training time while maintaining accuracy)


1. K-NEAREST NEIGHBORS CLASSIFIER

Predicting on validation set...
KNN: Predicting 10000 samples in batches...
  Processing samples 0/10000...
  Processing samples 5000/10000...

KNN - CLASSIFICATION REPORT
Class                Precision    Recall       F1-Score     Support   
--------------------------------------------------------------------------------
Action                   0.3902      0.4324      0.4103       407
Adventure                0.2786      0.3514      0.3108       111
Animation                0.4311      0.5079      0.4664       567
Comedy                   0.4186      0.4798      0.4471      1136
Crime                    0.3608      0.2953      0.3248       193
Documentary              0.5430      0.6302      0.5834      1563
Drama                    0.4

: 

In [9]:
import numpy as np
import pandas as pd
import time
from collections import Counter

# ============================================================
# EVALUATION METRICS (REUSED)
# ============================================================

def confusion_matrix_numpy(y_true, y_pred, num_classes):
    """Compute confusion matrix"""
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    return cm

def precision_recall_f1_numpy(y_true, y_pred, num_classes):
    """Compute precision, recall, and F1 score"""
    cm = confusion_matrix_numpy(y_true, y_pred, num_classes)
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
    
    return precision, recall, f1

def print_classification_report(y_true, y_pred, class_names, model_name="Model"):
    """Print classification report"""
    num_classes = len(class_names)
    precision, recall, f1 = precision_recall_f1_numpy(y_true, y_pred, num_classes)
    
    print("\n" + "="*80)
    print(f"{model_name} - CLASSIFICATION REPORT")
    print("="*80)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)
    
    for i in range(num_classes):
        support = np.sum(y_true == i)
        print(f"{class_names[i]:<20} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support:>8}")
    
    print("-"*80)
    
    # Macro & Weighted averages
    weights = np.array([np.sum(y_true == i) for i in range(num_classes)])
    weights = weights / np.sum(weights)
    
    print(f"{'Macro Avg':<20} {np.mean(precision):>10.4f}  {np.mean(recall):>10.4f}  "
          f"{np.mean(f1):>10.4f}  {len(y_true):>8}")
    print(f"{'Weighted Avg':<20} {np.sum(precision*weights):>10.4f}  "
          f"{np.sum(recall*weights):>10.4f}  {np.sum(f1*weights):>10.4f}  {len(y_true):>8}")
    print("="*80)
    print(f"Overall Accuracy: {np.mean(y_true == y_pred):.4f}")
    print("="*80)


# ============================================================
# 1. K-NEAREST NEIGHBORS (OPTIMIZED)
# ============================================================

class KNNClassifier:
    """
    Memory-efficient KNN using batch processing
    """
    
    def __init__(self, n_neighbors=5, batch_size=1000):
        self.k = n_neighbors
        self.batch_size = batch_size
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """Store training data"""
        self.X_train = X.astype(np.float32)  # Use float32 to save memory
        self.y_train = y
        return self
    
    def predict(self, X):
        """Predict using batch processing to save memory"""
        n_samples = X.shape[0]
        predictions = np.zeros(n_samples, dtype=np.int16)
        
        print(f"KNN: Predicting {n_samples} samples in batches...")
        
        for i in range(0, n_samples, self.batch_size):
            if i % 5000 == 0:
                print(f"  Processing samples {i}/{n_samples}...")
            
            end_idx = min(i + self.batch_size, n_samples)
            X_batch = X[i:end_idx]
            
            # Compute distances for this batch
            # Using euclidean distance: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a*b
            X_batch_sq = np.sum(X_batch**2, axis=1, keepdims=True)
            X_train_sq = np.sum(self.X_train**2, axis=1, keepdims=True).T
            distances = X_batch_sq + X_train_sq - 2 * np.dot(X_batch, self.X_train.T)
            
            # Find k nearest neighbors
            k_indices = np.argpartition(distances, self.k, axis=1)[:, :self.k]
            
            # Vote for each sample in batch
            for j in range(len(X_batch)):
                neighbor_labels = self.y_train[k_indices[j]]
                predictions[i + j] = np.bincount(neighbor_labels).argmax()
        
        return predictions


# ============================================================
# 2. DECISION TREE (OPTIMIZED)
# ============================================================

class DecisionTreeNode:
    """Node in decision tree"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # For leaf nodes
        
    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """
    Decision Tree with optimizations for speed
    """
    
    def __init__(self, max_depth=10, min_samples_split=20, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None
        self.n_classes = None
        
    def fit(self, X, y):
        """Build decision tree"""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        
        if self.max_features is None:
            self.max_features = int(np.sqrt(self.n_features))
        
        print(f"Building decision tree (max_depth={self.max_depth})...")
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity"""
        classes, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _best_split(self, X, y):
        """Find best split using random feature subset"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        # Randomly select features to consider
        features = np.random.choice(self.n_features, self.max_features, replace=False)
        
        parent_impurity = self._gini_impurity(y)
        
        for feature in features:
            # Sample thresholds instead of checking all unique values
            feature_values = X[:, feature]
            thresholds = np.percentile(feature_values, [25, 50, 75])
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_split or \
                   np.sum(right_mask) < self.min_samples_split:
                    continue
                
                # Calculate information gain
                left_impurity = self._gini_impurity(y[left_mask])
                right_impurity = self._gini_impurity(y[right_mask])
                
                n = len(y)
                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                
                gain = parent_impurity - (n_left/n * left_impurity + n_right/n * right_impurity)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth):
        """Recursively build tree"""
        n_samples = len(y)
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Find best split
        feature, threshold = self._best_split(X, y)
        
        if feature is None:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Split data
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # Build subtrees
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return DecisionTreeNode(feature=feature, threshold=threshold,
                               left=left_child, right=right_child)
    
    def _predict_sample(self, x, node):
        """Predict single sample"""
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict multiple samples"""
        print(f"Decision Tree: Predicting {len(X)} samples...")
        return np.array([self._predict_sample(x, self.root) for x in X])


# ============================================================
# 3. RANDOM FOREST (OPTIMIZED)
# ============================================================

class RandomForestClassifier:
    """
    Random Forest using bootstrap sampling and feature randomness
    """
    
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=20, 
                 max_features=None, sample_ratio=0.8):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.sample_ratio = sample_ratio
        self.trees = []
    
    def fit(self, X, y):
        """Train random forest"""
        n_samples = X.shape[0]
        sample_size = int(n_samples * self.sample_ratio)
        
        print(f"\nTraining Random Forest with {self.n_estimators} trees...")
        print(f"Each tree trains on {sample_size:,} samples")
        print("-" * 80)
        
        for i in range(self.n_estimators):
            print(f"Training tree {i+1}/{self.n_estimators}...")
            
            # Bootstrap sample
            indices = np.random.choice(n_samples, sample_size, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]
            
            # Train decision tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
        
        print("Random Forest training complete!")
        return self
    
    def predict(self, X):
        """Predict by majority voting"""
        print(f"\nRandom Forest: Generating predictions from {self.n_estimators} trees...")
        
        # Get predictions from all trees
        tree_predictions = np.zeros((len(X), self.n_estimators), dtype=np.int16)
        
        for i, tree in enumerate(self.trees):
            print(f"  Tree {i+1}/{self.n_estimators} predicting...")
            tree_predictions[:, i] = tree.predict(X)
        
        # Majority vote
        print("  Computing majority vote...")
        predictions = np.array([np.bincount(tree_predictions[i]).argmax() 
                               for i in range(len(X))])
        
        return predictions


# ============================================================
# MAIN EXECUTION WITH SAMPLING FOR SPEED
# ============================================================

print("\n" + "="*80)
print("TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS")
print("="*80)

# Increased sample sizes for better accuracy
TRAIN_SAMPLE_SIZE = 150000  # 150k samples for training (3x more)
TEST_SAMPLE_SIZE = 20000    # 20k samples for testing (2x more)

print(f"\nOriginal sizes: Train={len(X_train_clf):,}, Test={len(X_test_clf):,}")
print(f"Using samples: Train={TRAIN_SAMPLE_SIZE:,}, Test={TEST_SAMPLE_SIZE:,}")
print("(Balanced sample size for better accuracy and reasonable training time)\n")

# Sample training data
np.random.seed(42)
train_idx = np.random.choice(len(X_train_clf), TRAIN_SAMPLE_SIZE, replace=False)
X_train_sample = X_train_clf[train_idx]
y_train_sample = y_genre_train_clf[train_idx]

# Sample test data for evaluation
test_idx = np.random.choice(len(X_test_clf), TEST_SAMPLE_SIZE, replace=False)
X_test_sample = X_test_clf[test_idx]

# For evaluation, sample larger validation set
val_idx = np.random.choice(len(X_train_clf), 20000, replace=False)
X_val = X_train_clf[val_idx]
y_val = y_genre_train_clf[val_idx]

print("="*80)

# ============================================================
# 1. K-NEAREST NEIGHBORS
# ============================================================

print("\n" + "="*80)
print("1. K-NEAREST NEIGHBORS CLASSIFIER")
print("="*80)

start_time = time.time()

knn = KNNClassifier(n_neighbors=7, batch_size=1000)  # Increased K for better accuracy
knn.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_knn = knn.predict(X_val)

knn_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_knn, genre_list_clf, "KNN")
print(f"\nKNN Training + Prediction Time: {knn_time/60:.2f} minutes")

# ============================================================
# 2. DECISION TREE
# ============================================================

print("\n" + "="*80)
print("2. DECISION TREE CLASSIFIER")
print("="*80)

start_time = time.time()

dt = DecisionTreeClassifier(max_depth=20, min_samples_split=30, max_features=150)  # Deeper tree, more features
dt.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_dt = dt.predict(X_val)

dt_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_dt, genre_list_clf, "Decision Tree")
print(f"\nDecision Tree Training + Prediction Time: {dt_time/60:.2f} minutes")

# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "="*80)
print("3. RANDOM FOREST CLASSIFIER")
print("="*80)

start_time = time.time()

rf = RandomForestClassifier(
    n_estimators=20,      # 20 trees for better accuracy
    max_depth=18,         # Deeper trees
    min_samples_split=30,
    max_features=150,     # More features per split
    sample_ratio=0.8      # More data per tree
)
rf.fit(X_train_sample, y_train_sample)

print("\nPredicting on validation set...")
y_val_pred_rf = rf.predict(X_val)

rf_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_rf, genre_list_clf, "Random Forest")
print(f"\nRandom Forest Training + Prediction Time: {rf_time/60:.2f} minutes")

# ============================================================
# COMPARISON SUMMARY
# ============================================================

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

models_summary = {
    'Model': ['KNN', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        np.mean(y_val == y_val_pred_knn),
        np.mean(y_val == y_val_pred_dt),
        np.mean(y_val == y_val_pred_rf)
    ],
    'Time (min)': [knn_time/60, dt_time/60, rf_time/60]
}

# Calculate F1 scores
knn_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_knn, len(genre_list_clf))[2])
dt_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_dt, len(genre_list_clf))[2])
rf_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_rf, len(genre_list_clf))[2])

models_summary['Macro F1'] = [knn_f1, dt_f1, rf_f1]

summary_df = pd.DataFrame(models_summary)
print("\n", summary_df.to_string(index=False))
print("\n" + "="*80)

print("\n✅ ALL MODELS TRAINED AND EVALUATED SUCCESSFULLY!")
print("="*80)


TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS

Original sizes: Train=442,642, Test=144,464
Using samples: Train=150,000, Test=20,000
(Balanced sample size for better accuracy and reasonable training time)


1. K-NEAREST NEIGHBORS CLASSIFIER

Predicting on validation set...
KNN: Predicting 20000 samples in batches...
  Processing samples 0/20000...
  Processing samples 5000/20000...
  Processing samples 10000/20000...
  Processing samples 15000/20000...

KNN - CLASSIFICATION REPORT
Class                Precision    Recall       F1-Score     Support   
--------------------------------------------------------------------------------
Action                   0.2555      0.2204      0.2366       794
Adventure                0.1081      0.0727      0.0870       220
Animation                0.3097      0.3066      0.3081      1083
Comedy                   0.2781      0.3352      0.3040      2273
Crime                    0.1600      0.0588      0.0860       340
Documentary       

KeyboardInterrupt: 

In [10]:
import numpy as np
import pandas as pd
import time
from collections import Counter

# ============================================================
# EVALUATION METRICS (REUSED)
# ============================================================

def confusion_matrix_numpy(y_true, y_pred, num_classes):
    """Compute confusion matrix"""
    cm = np.zeros((num_classes, num_classes), dtype=np.int32)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    return cm

def precision_recall_f1_numpy(y_true, y_pred, num_classes):
    """Compute precision, recall, and F1 score"""
    cm = confusion_matrix_numpy(y_true, y_pred, num_classes)
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = cm[i, i]
        fp = np.sum(cm[:, i]) - tp
        fn = np.sum(cm[i, :]) - tp
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
    
    return precision, recall, f1

def print_classification_report(y_true, y_pred, class_names, model_name="Model"):
    """Print classification report"""
    num_classes = len(class_names)
    precision, recall, f1 = precision_recall_f1_numpy(y_true, y_pred, num_classes)
    
    print("\n" + "="*80)
    print(f"{model_name} - CLASSIFICATION REPORT")
    print("="*80)
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
    print("-"*80)
    
    for i in range(num_classes):
        support = np.sum(y_true == i)
        print(f"{class_names[i]:<20} {precision[i]:>10.4f}  {recall[i]:>10.4f}  "
              f"{f1[i]:>10.4f}  {support:>8}")
    
    print("-"*80)
    
    # Macro & Weighted averages
    weights = np.array([np.sum(y_true == i) for i in range(num_classes)])
    weights = weights / np.sum(weights)
    
    print(f"{'Macro Avg':<20} {np.mean(precision):>10.4f}  {np.mean(recall):>10.4f}  "
          f"{np.mean(f1):>10.4f}  {len(y_true):>8}")
    print(f"{'Weighted Avg':<20} {np.sum(precision*weights):>10.4f}  "
          f"{np.sum(recall*weights):>10.4f}  {np.sum(f1*weights):>10.4f}  {len(y_true):>8}")
    print("="*80)
    print(f"Overall Accuracy: {np.mean(y_true == y_pred):.4f}")
    print("="*80)


# ============================================================
# 1. K-NEAREST NEIGHBORS (OPTIMIZED)
# ============================================================

class KNNClassifier:
    """
    Memory-efficient KNN using batch processing
    """
    
    def __init__(self, n_neighbors=5, batch_size=1000):
        self.k = n_neighbors
        self.batch_size = batch_size
        self.X_train = None
        self.y_train = None
    
    def fit(self, X, y):
        """Store training data"""
        self.X_train = X.astype(np.float32)  # Use float32 to save memory
        self.y_train = y
        return self
    
    def predict(self, X):
        """Predict using batch processing to save memory"""
        n_samples = X.shape[0]
        predictions = np.zeros(n_samples, dtype=np.int16)
        
        print(f"KNN: Predicting {n_samples} samples in batches...")
        
        for i in range(0, n_samples, self.batch_size):
            if i % 5000 == 0:
                print(f"  Processing samples {i}/{n_samples}...")
            
            end_idx = min(i + self.batch_size, n_samples)
            X_batch = X[i:end_idx]
            
            # Compute distances for this batch
            # Using euclidean distance: ||a - b||^2 = ||a||^2 + ||b||^2 - 2*a*b
            X_batch_sq = np.sum(X_batch**2, axis=1, keepdims=True)
            X_train_sq = np.sum(self.X_train**2, axis=1, keepdims=True).T
            distances = X_batch_sq + X_train_sq - 2 * np.dot(X_batch, self.X_train.T)
            
            # Find k nearest neighbors
            k_indices = np.argpartition(distances, self.k, axis=1)[:, :self.k]
            
            # Vote for each sample in batch
            for j in range(len(X_batch)):
                neighbor_labels = self.y_train[k_indices[j]]
                predictions[i + j] = np.bincount(neighbor_labels).argmax()
        
        return predictions


# ============================================================
# 2. DECISION TREE (OPTIMIZED)
# ============================================================

class DecisionTreeNode:
    """Node in decision tree"""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # For leaf nodes
        
    def is_leaf(self):
        return self.value is not None


class DecisionTreeClassifier:
    """
    Decision Tree with optimizations for speed
    """
    
    def __init__(self, max_depth=10, min_samples_split=20, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None
        self.n_classes = None
        
    def fit(self, X, y):
        """Build decision tree"""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        
        if self.max_features is None:
            self.max_features = int(np.sqrt(self.n_features))
        
        print(f"Building decision tree (max_depth={self.max_depth})...")
        self.root = self._build_tree(X, y, depth=0)
        return self
    
    def _gini_impurity(self, y):
        """Calculate Gini impurity"""
        classes, counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        return 1 - np.sum(probabilities ** 2)
    
    def _best_split(self, X, y):
        """Find best split using random feature subset"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        # Randomly select features to consider
        features = np.random.choice(self.n_features, self.max_features, replace=False)
        
        parent_impurity = self._gini_impurity(y)
        
        for feature in features:
            # Sample thresholds instead of checking all unique values
            feature_values = X[:, feature]
            thresholds = np.percentile(feature_values, [25, 50, 75])
            
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_split or \
                   np.sum(right_mask) < self.min_samples_split:
                    continue
                
                # Calculate information gain
                left_impurity = self._gini_impurity(y[left_mask])
                right_impurity = self._gini_impurity(y[right_mask])
                
                n = len(y)
                n_left = np.sum(left_mask)
                n_right = np.sum(right_mask)
                
                gain = parent_impurity - (n_left/n * left_impurity + n_right/n * right_impurity)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold
    
    def _build_tree(self, X, y, depth):
        """Recursively build tree"""
        n_samples = len(y)
        
        # Stopping criteria
        if depth >= self.max_depth or n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Find best split
        feature, threshold = self._best_split(X, y)
        
        if feature is None:
            leaf_value = np.bincount(y).argmax()
            return DecisionTreeNode(value=leaf_value)
        
        # Split data
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # Build subtrees
        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return DecisionTreeNode(feature=feature, threshold=threshold,
                               left=left_child, right=right_child)
    
    def _predict_sample(self, x, node):
        """Predict single sample"""
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict multiple samples"""
        print(f"Decision Tree: Predicting {len(X)} samples...")
        return np.array([self._predict_sample(x, self.root) for x in X])


# ============================================================
# 3. RANDOM FOREST (OPTIMIZED)
# ============================================================

class RandomForestClassifier:
    """
    Random Forest using bootstrap sampling and feature randomness
    """
    
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=20, 
                 max_features=None, sample_ratio=0.8):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.sample_ratio = sample_ratio
        self.trees = []
    
    def fit(self, X, y):
        """Train random forest"""
        n_samples = X.shape[0]
        sample_size = int(n_samples * self.sample_ratio)
        
        print(f"\nTraining Random Forest with {self.n_estimators} trees...")
        print(f"Each tree trains on {sample_size:,} samples")
        print("-" * 80)
        
        for i in range(self.n_estimators):
            print(f"Training tree {i+1}/{self.n_estimators}...")
            
            # Bootstrap sample
            indices = np.random.choice(n_samples, sample_size, replace=True)
            X_sample = X[indices]
            y_sample = y[indices]
            
            # Train decision tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
        
        print("Random Forest training complete!")
        return self
    
    def predict(self, X):
        """Predict by majority voting"""
        print(f"\nRandom Forest: Generating predictions from {self.n_estimators} trees...")
        
        # Get predictions from all trees
        tree_predictions = np.zeros((len(X), self.n_estimators), dtype=np.int16)
        
        for i, tree in enumerate(self.trees):
            print(f"  Tree {i+1}/{self.n_estimators} predicting...")
            tree_predictions[:, i] = tree.predict(X)
        
        # Majority vote
        print("  Computing majority vote...")
        predictions = np.array([np.bincount(tree_predictions[i]).argmax() 
                               for i in range(len(X))])
        
        return predictions


# ============================================================
# MAIN EXECUTION - FULL DATASET
# ============================================================

print("\n" + "="*80)
print("TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS")
print("="*80)

# USE FULL DATASET FOR MAXIMUM ACCURACY
print(f"\n🔥 RUNNING ON FULL DATASET 🔥")
print(f"Training samples: {len(X_train_clf):,}")
print(f"Test samples: {len(X_test_clf):,}")
print("\nNote: This will take longer but provide best accuracy!")
print("Estimated time: 15-30 minutes depending on your machine")
print("="*80)

# Use full training data
X_train_full = X_train_clf
y_train_full = y_genre_train_clf

# Create validation set (20k samples for faster evaluation)
np.random.seed(42)
val_idx = np.random.choice(len(X_train_clf), 30000, replace=False)
X_val = X_train_clf[val_idx]
y_val = y_genre_train_clf[val_idx]

print(f"\nValidation set: {len(X_val):,} samples")
print("="*80)

# ============================================================
# 1. K-NEAREST NEIGHBORS
# ============================================================

print("\n" + "="*80)
print("1. K-NEAREST NEIGHBORS CLASSIFIER")
print("="*80)

start_time = time.time()

knn = KNNClassifier(n_neighbors=9, batch_size=2000)  # Larger batch for speed
print(f"Training KNN on full dataset ({len(X_train_full):,} samples)...")
knn.fit(X_train_full, y_train_full)

print("\nPredicting on validation set...")
y_val_pred_knn = knn.predict(X_val)

knn_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_knn, genre_list_clf, "KNN")
print(f"\nKNN Training + Prediction Time: {knn_time/60:.2f} minutes")

# ============================================================
# 2. DECISION TREE
# ============================================================

print("\n" + "="*80)
print("2. DECISION TREE CLASSIFIER")
print("="*80)

start_time = time.time()

dt = DecisionTreeClassifier(max_depth=25, min_samples_split=50, max_features=200)
print(f"Training Decision Tree on full dataset ({len(X_train_full):,} samples)...")
dt.fit(X_train_full, y_train_full)

print("\nPredicting on validation set...")
y_val_pred_dt = dt.predict(X_val)

dt_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_dt, genre_list_clf, "Decision Tree")
print(f"\nDecision Tree Training + Prediction Time: {dt_time/60:.2f} minutes")

# ============================================================
# 3. RANDOM FOREST
# ============================================================

print("\n" + "="*80)
print("3. RANDOM FOREST CLASSIFIER")
print("="*80)

start_time = time.time()

rf = RandomForestClassifier(
    n_estimators=15,      # 15 trees for balance
    max_depth=20,         # Deep trees
    min_samples_split=100, # Higher for large dataset
    max_features=200,     # More features
    sample_ratio=0.5      # 50% sampling for speed with large dataset
)
print(f"Training Random Forest on full dataset ({len(X_train_full):,} samples)...")
print("This will take the longest (10-20 minutes estimated)...")
rf.fit(X_train_full, y_train_full)

print("\nPredicting on validation set...")
y_val_pred_rf = rf.predict(X_val)

rf_time = time.time() - start_time

print_classification_report(y_val, y_val_pred_rf, genre_list_clf, "Random Forest")
print(f"\nRandom Forest Training + Prediction Time: {rf_time/60:.2f} minutes")

# ============================================================
# COMPARISON SUMMARY
# ============================================================

print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

models_summary = {
    'Model': ['KNN', 'Decision Tree', 'Random Forest'],
    'Accuracy': [
        np.mean(y_val == y_val_pred_knn),
        np.mean(y_val == y_val_pred_dt),
        np.mean(y_val == y_val_pred_rf)
    ],
    'Time (min)': [knn_time/60, dt_time/60, rf_time/60]
}

# Calculate F1 scores
knn_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_knn, len(genre_list_clf))[2])
dt_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_dt, len(genre_list_clf))[2])
rf_f1 = np.mean(precision_recall_f1_numpy(y_val, y_val_pred_rf, len(genre_list_clf))[2])

models_summary['Macro F1'] = [knn_f1, dt_f1, rf_f1]

summary_df = pd.DataFrame(models_summary)
print("\n", summary_df.to_string(index=False))
print("\n" + "="*80)

print("\n✅ ALL MODELS TRAINED AND EVALUATED SUCCESSFULLY!")
print("="*80)


TRAINING KNN, DECISION TREE, AND RANDOM FOREST CLASSIFIERS

🔥 RUNNING ON FULL DATASET 🔥
Training samples: 442,642
Test samples: 144,464

Note: This will take longer but provide best accuracy!
Estimated time: 15-30 minutes depending on your machine

Validation set: 30,000 samples

1. K-NEAREST NEIGHBORS CLASSIFIER
Training KNN on full dataset (442,642 samples)...

Predicting on validation set...
KNN: Predicting 30000 samples in batches...
  Processing samples 0/30000...
  Processing samples 10000/30000...
  Processing samples 20000/30000...

KNN - CLASSIFICATION REPORT
Class                Precision    Recall       F1-Score     Support   
--------------------------------------------------------------------------------
Action                   0.3784      0.3076      0.3393      1138
Adventure                0.2658      0.1138      0.1594       369
Animation                0.4291      0.4072      0.4179      1628
Comedy                   0.3883      0.4407      0.4128      3422
Crime   

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# ============================================================
# K-FOLD CROSS VALIDATION FROM SCRATCH
# ============================================================

class KFoldCV:
    """K-Fold Cross Validation implementation"""
    
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.n_splits = n_splits
        self.shuffle = shuffle
        self.random_state = random_state
    
    def split(self, X, y):
        """Generate train/test indices for k-fold"""
        n_samples = len(X)
        indices = np.arange(n_samples)
        
        if self.shuffle:
            np.random.seed(self.random_state)
            np.random.shuffle(indices)
        
        fold_size = n_samples // self.n_splits
        
        for i in range(self.n_splits):
            start = i * fold_size
            end = start + fold_size if i < self.n_splits - 1 else n_samples
            
            test_idx = indices[start:end]
            train_idx = np.concatenate([indices[:start], indices[end:]])
            
            yield train_idx, test_idx


def evaluate_model(y_true, y_pred, num_classes):
    """Compute accuracy, precision, recall, f1"""
    accuracy = np.mean(y_true == y_pred)
    
    # Per-class metrics
    precision = np.zeros(num_classes)
    recall = np.zeros(num_classes)
    f1 = np.zeros(num_classes)
    
    for i in range(num_classes):
        tp = np.sum((y_true == i) & (y_pred == i))
        fp = np.sum((y_true != i) & (y_pred == i))
        fn = np.sum((y_true == i) & (y_pred != i))
        
        precision[i] = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall[i] = tp / (tp + fn) if tp + fn > 0 else 0.0
        
        if precision[i] + recall[i] > 0:
            f1[i] = 2 * (precision[i] * recall[i]) / (precision[i] + recall[i])
    
    return {
        'accuracy': accuracy,
        'precision': np.mean(precision),
        'recall': np.mean(recall),
        'f1': np.mean(f1)
    }


def cross_validate(model, X, y, cv, model_name, num_classes):
    """Perform cross-validation on a model"""
    scores = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'train_time': [],
        'test_time': []
    }
    
    print(f"\n{'='*80}")
    print(f"K-FOLD CROSS-VALIDATION: {model_name}")
    print(f"{'='*80}")
    print(f"Dataset size: {len(X):,} | Folds: {cv.n_splits}")
    print(f"{'='*80}")
    
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        print(f"\nFold {fold_idx + 1}/{cv.n_splits}:")
        print(f"  Train samples: {len(train_idx):,} | Test samples: {len(test_idx):,}")
        
        X_train_fold = X[train_idx]
        y_train_fold = y[train_idx]
        X_test_fold = X[test_idx]
        y_test_fold = y[test_idx]
        
        # Train
        start_time = time.time()
        model.fit(X_train_fold, y_train_fold)
        train_time = time.time() - start_time
        
        # Test
        start_time = time.time()
        y_pred = model.predict(X_test_fold)
        test_time = time.time() - start_time
        
        # Evaluate
        metrics = evaluate_model(y_test_fold, y_pred, num_classes)
        
        scores['accuracy'].append(metrics['accuracy'])
        scores['precision'].append(metrics['precision'])
        scores['recall'].append(metrics['recall'])
        scores['f1'].append(metrics['f1'])
        scores['train_time'].append(train_time)
        scores['test_time'].append(test_time)
        
        print(f"  Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f} | "
              f"Train time: {train_time/60:.2f}min | Test time: {test_time/60:.2f}min")
    
    # Summary
    print(f"\n{'-'*80}")
    print(f"CROSS-VALIDATION SUMMARY - {model_name}")
    print(f"{'-'*80}")
    print(f"Accuracy:  {np.mean(scores['accuracy']):.4f} ± {np.std(scores['accuracy']):.4f}")
    print(f"Precision: {np.mean(scores['precision']):.4f} ± {np.std(scores['precision']):.4f}")
    print(f"Recall:    {np.mean(scores['recall']):.4f} ± {np.std(scores['recall']):.4f}")
    print(f"F1-Score:  {np.mean(scores['f1']):.4f} ± {np.std(scores['f1']):.4f}")
    print(f"Avg Train Time: {np.mean(scores['train_time'])/60:.2f} min")
    print(f"Avg Test Time:  {np.mean(scores['test_time'])/60:.2f} min")
    print(f"{'='*80}")
    
    return scores


# ============================================================
# IMPORT MODELS (Use from previous implementation)
# ============================================================

# Assuming LogisticRegressionMulticlass, KNNClassifier, 
# DecisionTreeClassifier, RandomForestClassifier are defined

# ============================================================
# RUN CROSS-VALIDATION ON SAMPLED DATA
# ============================================================

print("\n" + "="*80)
print("K-FOLD CROSS-VALIDATION FOR ALL CLASSIFIERS")
print("="*80)

# Use sample for CV to keep it reasonable (CV multiplies training time by k)
CV_SAMPLE_SIZE = 100000  # 100k samples for CV
print(f"\nUsing {CV_SAMPLE_SIZE:,} samples for cross-validation")
print("(Full dataset would take too long for 5-fold CV)")
print("="*80)

np.random.seed(42)
cv_idx = np.random.choice(len(X_train_clf), CV_SAMPLE_SIZE, replace=False)
X_cv = X_train_clf[cv_idx]
y_cv = y_genre_train_clf[cv_idx]

# Initialize cross-validator
kfold = KFoldCV(n_splits=5, shuffle=True, random_state=42)

# Store all results
all_cv_results = {}

# ============================================================
# 1. LOGISTIC REGRESSION CV
# ============================================================

print("\n" + "="*80)
print("1. LOGISTIC REGRESSION - CROSS VALIDATION")
print("="*80)

lr_model = LogisticRegressionMulticlass(
    learning_rate=0.8,
    num_iterations=50,
    batch_size=512,
    l2_lambda=0.005,
    verbose=False  # Turn off verbose for CV
)

lr_scores = cross_validate(lr_model, X_cv, y_cv, kfold, 
                           "Logistic Regression", len(genre_list_clf))
all_cv_results['Logistic Regression'] = lr_scores

# ============================================================
# 2. KNN CV
# ============================================================

print("\n" + "="*80)
print("2. K-NEAREST NEIGHBORS - CROSS VALIDATION")
print("="*80)

knn_model = KNNClassifier(n_neighbors=9, batch_size=2000)

knn_scores = cross_validate(knn_model, X_cv, y_cv, kfold, 
                            "K-Nearest Neighbors", len(genre_list_clf))
all_cv_results['KNN'] = knn_scores

# ============================================================
# 3. DECISION TREE CV
# ============================================================

print("\n" + "="*80)
print("3. DECISION TREE - CROSS VALIDATION")
print("="*80)

dt_model = DecisionTreeClassifier(max_depth=20, min_samples_split=50, max_features=150)

dt_scores = cross_validate(dt_model, X_cv, y_cv, kfold, 
                           "Decision Tree", len(genre_list_clf))
all_cv_results['Decision Tree'] = dt_scores

# ============================================================
# 4. RANDOM FOREST CV
# ============================================================

print("\n" + "="*80)
print("4. RANDOM FOREST - CROSS VALIDATION")
print("="*80)

rf_model = RandomForestClassifier(
    n_estimators=10,
    max_depth=18,
    min_samples_split=50,
    max_features=150,
    sample_ratio=0.6
)

rf_scores = cross_validate(rf_model, X_cv, y_cv, kfold, 
                           "Random Forest", len(genre_list_clf))
all_cv_results['Random Forest'] = rf_scores

# ============================================================
# VISUALIZATION: CROSS-VALIDATION RESULTS
# ============================================================

print("\n" + "="*80)
print("GENERATING VISUALIZATION GRAPHS")
print("="*80)

# Prepare data for plotting
models = list(all_cv_results.keys())
metrics = ['accuracy', 'precision', 'recall', 'f1']

# Create figure with subplots
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('K-Fold Cross-Validation Results - All Models', fontsize=16, fontweight='bold')

# 1. Accuracy Comparison
ax = axes[0, 0]
accuracies = [all_cv_results[model]['accuracy'] for model in models]
positions = np.arange(len(models))
bp = ax.boxplot(accuracies, labels=models, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Across Folds', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(models, rotation=45, ha='right')

# 2. F1-Score Comparison
ax = axes[0, 1]
f1_scores = [all_cv_results[model]['f1'] for model in models]
bp = ax.boxplot(f1_scores, labels=models, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightgreen')
ax.set_ylabel('F1-Score', fontsize=12)
ax.set_title('F1-Score Across Folds', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(models, rotation=45, ha='right')

# 3. Precision vs Recall
ax = axes[0, 2]
for model in models:
    precision_mean = np.mean(all_cv_results[model]['precision'])
    recall_mean = np.mean(all_cv_results[model]['recall'])
    ax.scatter(recall_mean, precision_mean, s=200, alpha=0.6, label=model)
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision vs Recall', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# 4. Mean Performance Metrics
ax = axes[1, 0]
x = np.arange(len(models))
width = 0.2

for i, metric in enumerate(metrics):
    means = [np.mean(all_cv_results[model][metric]) for model in models]
    ax.bar(x + i*width, means, width, label=metric.capitalize(), alpha=0.8)

ax.set_xlabel('Models', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Mean Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# 5. Training Time Comparison
ax = axes[1, 1]
train_times = [np.mean(all_cv_results[model]['train_time'])/60 for model in models]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bars = ax.bar(models, train_times, color=colors, alpha=0.7)
ax.set_ylabel('Time (minutes)', fontsize=12)
ax.set_title('Average Training Time per Fold', fontsize=14, fontweight='bold')
ax.set_xticklabels(models, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}min',
            ha='center', va='bottom', fontsize=10)

# 6. Performance Stability (Standard Deviation)
ax = axes[1, 2]
stds = {model: np.std(all_cv_results[model]['accuracy']) for model in models}
bars = ax.bar(models, list(stds.values()), color='coral', alpha=0.7)
ax.set_ylabel('Standard Deviation', fontsize=12)
ax.set_title('Model Stability (Lower is Better)', fontsize=14, fontweight='bold')
ax.set_xticklabels(models, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('cross_validation_results.png', dpi=300, bbox_inches='tight')
print("✅ Saved: cross_validation_results.png")
plt.show()

# ============================================================
# ADDITIONAL GRAPH: Learning Curves Per Model
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Performance Across Folds - Individual Models', fontsize=16, fontweight='bold')

for idx, (model, scores) in enumerate(all_cv_results.items()):
    ax = axes[idx // 2, idx % 2]
    
    folds = np.arange(1, 6)
    
    ax.plot(folds, scores['accuracy'], marker='o', linewidth=2, markersize=8, 
            label='Accuracy', color='blue')
    ax.plot(folds, scores['precision'], marker='s', linewidth=2, markersize=8, 
            label='Precision', color='green')
    ax.plot(folds, scores['recall'], marker='^', linewidth=2, markersize=8, 
            label='Recall', color='orange')
    ax.plot(folds, scores['f1'], marker='d', linewidth=2, markersize=8, 
            label='F1-Score', color='red')
    
    ax.set_xlabel('Fold Number', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(f'{model}', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)
    ax.set_xticks(folds)

plt.tight_layout()
plt.savefig('individual_fold_performance.png', dpi=300, bbox_inches='tight')
print("✅ Saved: individual_fold_performance.png")
plt.show()

# ============================================================
# SUMMARY TABLE
# ============================================================

print("\n" + "="*80)
print("FINAL CROSS-VALIDATION SUMMARY TABLE")
print("="*80)

summary_data = {
    'Model': models,
    'Accuracy (mean±std)': [f"{np.mean(all_cv_results[m]['accuracy']):.4f}±{np.std(all_cv_results[m]['accuracy']):.4f}" 
                            for m in models],
    'F1-Score (mean±std)': [f"{np.mean(all_cv_results[m]['f1']):.4f}±{np.std(all_cv_results[m]['f1']):.4f}" 
                            for m in models],
    'Precision (mean±std)': [f"{np.mean(all_cv_results[m]['precision']):.4f}±{np.std(all_cv_results[m]['precision']):.4f}" 
                             for m in models],
    'Recall (mean±std)': [f"{np.mean(all_cv_results[m]['recall']):.4f}±{np.std(all_cv_results[m]['recall']):.4f}" 
                          for m in models],
    'Avg Train Time (min)': [f"{np.mean(all_cv_results[m]['train_time'])/60:.2f}" for m in models]
}

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))
print("\n" + "="*80)

# Find best model
best_model_idx = np.argmax([np.mean(all_cv_results[m]['accuracy']) for m in models])
best_model = models[best_model_idx]
best_accuracy = np.mean(all_cv_results[best_model]['accuracy'])

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Accuracy: {best_accuracy:.4f}")
print(f"   F1-Score: {np.mean(all_cv_results[best_model]['f1']):.4f}")
print("="*80)

print("\n✅ CROSS-VALIDATION COMPLETE!")
print("📊 Graphs saved as PNG files in current directory")
print("="*80)


K-FOLD CROSS-VALIDATION FOR ALL CLASSIFIERS

Using 100,000 samples for cross-validation
(Full dataset would take too long for 5-fold CV)

1. LOGISTIC REGRESSION - CROSS VALIDATION

K-FOLD CROSS-VALIDATION: Logistic Regression
Dataset size: 100,000 | Folds: 5

Fold 1/5:
  Train samples: 80,000 | Test samples: 20,000
Training on 80000 samples with 3009 features
Number of classes: 20
Batch size: 512, Learning rate: 0.8
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Training completed in 5.06 minutes
  Accuracy: 0.3287 | F1: 0.0659 | Train time: 5.06min | Test time: 0.01min

Fold 2/5:
  Train samples: 80,000 | Test samples: 20,000
Training on 80000 samples with 3009 features
Number of classes: 20
Batch size: 512, Learning rate: 0.8
--------------------------------------------------------------------------------
-----------------------------------------------------------------